In [1]:
using Oscar
using ProgressBars
using SparseArrays
using Serialization


  ___   ___   ___    _    ____
 / _ \ / __\ / __\  / \  |  _ \  | Combining and extending ANTIC, GAP,
| |_| |\__ \| |__  / ^ \ |  ´ /  | Polymake and Singular
 \___/ \___/ \___//_/ \_\|_|\_\  | Type "?Oscar" for more information
o--------o-----o-----o--------o  | Documentation: https://docs.oscar-system.org
  S Y M B O L I C   T O O L S    | Version 1.4.1


In [ ]:
using Polymake
const topaz = Polymake.topaz
using ProgressMeter
using Nemo
F = GF(2)


# Build canonical vertex list present in a matroid (sorted)
function vertex_list_of_bases(bases::Vector{Vector{Unsigned}})
    s = Set{UInt8}()
    for b in bases
        for v in b
            push!(s, v)
        end
    end
    return sort(collect(s))
end

# Construct a matrix from a binary matroid

function binary_matrix_representation(M; B=nothing)
    is_binary(M) || error("Matroid is not binary")

    # ground set
    E = collect(matroid_groundset(M))
    n = length(E)

    # choose basis
    if B === nothing
        B = first(bases(M))
    else
        length(B) == rank(M) || error("Provided set is not a basis")
    end

    r = length(B)

    # index maps
    row_of = Dict(b => i for (i,b) in enumerate(B))
    col_of = Dict(e => j for (j,e) in enumerate(E))

    # initialize matrix
    A = matrix(F,zeros(Int, (r, n)))

    # basis columns = identity
    for b in B
        A[row_of[b], col_of[b]] = F(1)
    end

    # non-basis columns via fundamental circuits
    for e in E
        e in B && continue
        C = fundamental_circuit(M, B, e)
        for b in C
            b in B || continue
            A[row_of[b], col_of[e]] = F(1)
        end
    end

    return A, B, E
end

# ---------------------------
# using Topaz for isom
# ---------------------------

function bases_to_topaz_complex(bases::Vector{Vector{Unsigned}})
    # collect and sort vertices
    verts = sort!(unique(vcat(bases...)))

    # map vertex labels to 0-based indices
    vmap = Dict(v => i-1 for (i,v) in enumerate(verts))

    facets = [ [vmap[x] for x in B] for B in bases ]
    return topaz.SimplicialComplex(FACETS = facets), verts
end

function topaz_isomorphism(basesA, basesB)
    SC_A, vertsA = bases_to_topaz_complex(basesA)
    SC_B, vertsB = bases_to_topaz_complex(basesB)
    length(vertsA) == length(vertsB) || return nothing
    T = topaz.find_facet_vertex_permutations(SC_A, SC_B)
    if T===nothing
        return nothing
    end
    _,p=T
    perm = Dict(
        vertsA[i] => vertsB[p[i]+1]
        for i in eachindex(vertsA)
    )
    return perm
end






# ---------------------------
# Main builder
# ---------------------------
"""
build_iso_db!(Iso_DB, mat_DB; ms = sort(collect(keys(mat_DB))), verbose=false)

- Iso_DB will be mutated (create empty Dict before passing).
- mat_DB: Dict{Int, Vector{Vector{Vector{Int}}}} mapping m -> list of matroid-bases
- For each m in ms (skips if m-1 not present), and each k in mat_DB[m],
  finds first vertex v in 1:m such that corank(contract_at_vertex(M,v)) >= corank(M),
  computes contraction, then finds l and a permutation mapping contraction -> some mat_DB[m-1][l].
- Stores Iso_DB[m][k] = (l, mapping::Dict{Int,Int}) or (-1, nothing) if not found.
"""
function build_iso_db!(Iso_DB::Dict{Int,Dict{Int,Tuple{Int,Int,Any}}}, mat_DB::Dict{Int,Vector{Vector{Unsigned}}}; ms=nothing, verbose=false)
    # ms === nothing && (ms = sort(collect(keys(mat_DB))))
    for m in ms
        println(m)
        if !haskey(mat_DB, m-1)
            verbose && @info("Skipping m=$m because mat_DB[$(m-1)] not present")
            continue
        end
        Iso_DB[m] = Dict{Int,Tuple{Int,Int,Any}}()
        for (k, Mbases_bin) in enumerate(mat_DB[m])
            Mbases= [[UInt8(i) for i=1:16 if ((base_bin>>(i-1))&UInt16(1))==1] for base_bin in Mbases_bin]
            V=vertex_list_of_bases(Mbases)
            M = matroid_from_bases(Mbases,V)
            found_v = nothing
            Mv_chosen = nothing
            # pick smallest v in 1:m satisfying corank(contract) >= corank(M)
            v_chosen=nothing
            for v in V
                if v in coloops(M)
                    continue
                end
                Mv = deletion(M,v)
                if rank(Mv)==rank(M)
                    found_v=true
                    v_chosen=v
                    Mv_chosen=Mv
                    break
                end
            end
            if found_v === nothing
                v_chosen = V[1]
            end
            Mv_chosen = deletion(M,v_chosen)
            deletion_bases = bases(Mv_chosen)

            # search through mat_DB[m-1] for an isomorphism
            perm = nothing
            found_index = -1
            found_isom = false
            for (l, target_bases_bin) in enumerate(mat_DB[m-1])
                target_bases = [[UInt8(i) for i=1:16 if ((base_bin>>(i-1))&UInt16(1))==1] for base_bin in target_bases_bin]
                M2 = matroid_from_bases(target_bases,vertex_list_of_bases(target_bases))
                if !is_isomorphic(Mv_chosen,M2)
                    continue
                end
                perm = topaz_isomorphism(target_bases,deletion_bases)
                if perm !== nothing
                    found_index = l
                    break
                end
            end

            if perm === nothing
                Iso_DB[m][k] = (-1, v_chosen, nothing)
                A,_,_ = binary_matrix_representation(M)
                display(A)
                verbose && @warn(
                    "No isomorphic matroid found for contraction  m=$m, k=$k at v=$v_chosen"
                )
            else
                Iso_DB[m][k] = (found_index,v_chosen, perm)
                # verbose && @info(
                #     "Found iso: m=$m, k=$k -> m-1 index=$found_index, v=$v_chosen"
                # )
            end
        end
    end
    return Iso_DB
end


build_iso_db!

In [28]:
global mat_DB = open("rank_4_simple_bin_mat_DB_bin.jls", "r") do io
    deserialize(io)
end

iso_DB = Dict{Int,Dict{Int,Tuple{Int,Int,Any}}}()

build_iso_db!(iso_DB,mat_DB,ms=7:15,verbose=true)


open("rank_4_iso_DB_7-15_bin.jls", "w") do io
    serialize(io, iso_DB)
end

7
8
9
10
11
12
13
14
15
